In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter as rcts
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate

LOCAL_LLM = 'gemma4:e4b'
EMBEDDING_MODEL = 'nomic-embed-text:latest'
TEMPERATURE = 0.7

/var/folders/qp/9vxvmncx0ks8cprx94py8fdh0000gn/T/ipykernel_25422/3480396480.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory


In [2]:
chatbot = ChatOllama(model=LOCAL_LLM, temperature=TEMPERATURE)

In [3]:
# Web search and scrape.
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper as ddg
import requests
from bs4 import BeautifulSoup

class WebSearch:
    def __init__(self, timeout:int = 15):
        self._timeout = timeout
        self._headers = {
            'User-Agent': (
                'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                'AppleWebKit/537.36 (KHTML, like Gecko) '
                'Chrome/124.0.0.0 Safari/537.36'
            ),
            'Accept-Language': 'en-US,en;q=0.9',
        }    

    def search(
        self,
        web_query: str,
        num_results: int,
    ) -> Sequence[str]:
        return [
            result['link'] for result in ddg().results(
                web_query, num_results
            )
        ]
        
    def scrape(self, url: str) -> str:
        """Scrape a webpage."""
        try:
            response = requests.get(url, headers=self._headers, timeout=self._timeout)
            if response.status_code == 200:
                return BeautifulSoup(
                    response.text, 'html.parser'
                ).get_text(separator=' ', strip=True)
            else:
                return f"Failed to retrieve the webpage: Status code {response.status_code}"
        except Exception as e:
            print(e)
            return f"Failed to retrieve the webpage: {e}"
            

In [4]:
# Add some data to Vector Store.
links = [
    'https://en.wikipedia.org/wiki/Paestum',
    'https://www.britannica.com/place/Paestum',
    'https://www.walksofitaly.com/blog/art-culture/the-best-ancient-greek-ruins-in-italys-mainland-paestum',
    'http://home.moravian.edu/users/phys/mejjg01/retirement%20activities/pages/geo/149-Paestum.html',
    'https://www.travelamalficoast.it/en/paestum-the-splendor-of-ancient-greece-near-amalfi-coast',
]

ws = WebSearch()
texts = [ws.scrape(link) for link in links]
    
text_splitter = rcts(
    chunk_size=1000,
    chunk_overlap=100, # Overlap helps keep context
    separators=["\n\n", "\n", ". ", " ", ""], # Priority: Paragraphs -> Newlines -> Sentences
    length_function=len,
)

chunks = [chunk for text in texts for chunk in text_splitter.split_text(text)]
print(f'{len(chunks)} Chunks.')   

embeddings_model = OllamaEmbeddings(model=EMBEDDING_MODEL)
vector_db = Chroma('tourist_info', embeddings_model)
docs = [Document(page_content=chunk) for chunk in chunks]
_ = vector_db.add_documents(docs)

95 Chunks.


In [5]:
rag_prompt_template = '''
Use the following pieces of context to answer the question at the end. 
If you don't know the answer, just say that you don't know,  don't try to make up an answer.
Use three sentences maximum and keep the answer as concise as possible.
{context}
Question: {question}
Helpful Answer:
'''
rag_prompt = PromptTemplate.from_template(rag_prompt_template)

retriever = vector_db.as_retriever()
question_feeder = RunnablePassthrough()
rag_chain = {
    'context': retriever,
    'question': question_feeder
} | rag_prompt | chatbot


In [6]:
question = 'Where was Poseidonia and who renamed it to Paestum? Also tell me the source.'
answer = rag_chain.invoke(question)
print(answer.content)

The ancient city of Poseidonia is located in the modern frazione of Paestum, which is part of the comune of Capaccio Paestum in Salerno, Campania, Italy. The Romans took over the city in 273 BCE, renaming it Paestum and establishing a Latin colony. This information is sourced from documents 4ac58a7b-b48e-4e5a-a3f0-d4ae670eaa60 and cd44efd4-c7f5-4319-8810-4ecf2ff2bdfd.


In [7]:
question = 'And then, what did they do?'
answer = rag_chain.invoke(question)
print(answer.content)

I don't know. The question "And then, what did they do?" is too vague and does not refer to a specific action or preceding event mentioned in the provided context.


In [8]:
rag_prompt = ChatPromptTemplate.from_messages([
        ('system', '''
        You are a helpful assistant, world-class expert in Roman and Greek history,
        especially in towns located in southern Italy. Provide interesting insights
        on local history and recommend places to visit with knowledgeable and
        engaging answers. Answer all questions to the best of your ability,
        but only use what has been provided in the context. If you don't know,
        just say you don't know. Use three sentences maximum and keep the answer
        as concise as possible.'''
        ),
        ('placeholder', '{chat_history_messages}'),
        ('assistant', '{retrieved_context}'),
        ('human', '{question}'),
])

chat_history_memory = ChatMessageHistory()
def get_messages(x):
    return chat_history_memory.messages

rag_chain = {
    'retrieved_context': retriever, 
    'question': question_feeder,
    'chat_history_messages': RunnableLambda(get_messages)
} | rag_prompt | chatbot

def execute_chain_with_memory(chain, question):
    chat_history_memory.add_user_message(question)
    answer = chain.invoke(question)
    chat_history_memory.add_ai_message(answer)
    print(f'Full chat message history: {chat_history_memory.messages}\n\n')                                      
    return answer

In [9]:
question = 'Where was Poseidonia and who renamed it to Paestum?'
answer = execute_chain_with_memory(rag_chain, question)
print(answer.content)

Full chat message history: [HumanMessage(content='Where was Poseidonia and who renamed it to Paestum?', additional_kwargs={}, response_metadata={}), AIMessage(content='The remains of Poseidonia are located in the modern frazione of Paestum, which is part of the comune of Capaccio Paestum in the Campania region of Italy. While the Greek settlers originally founded it, the Lucanians renamed the city to Paistos. The Romans later took over in 273 BCE, giving the city its current name, Paestum.', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-06-01T23:17:30.226607Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10972387292, 'load_duration': 104391667, 'prompt_eval_count': 1006, 'prompt_eval_duration': 1271783625, 'eval_count': 632, 'eval_duration': 9442151157, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019e857a-3514-7250-9835-f0909542250d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'inpu

In [10]:
question = 'And then what did they do?'
answer = execute_chain_with_memory(rag_chain, question)
print(answer.content)

Full chat message history: [HumanMessage(content='Where was Poseidonia and who renamed it to Paestum?', additional_kwargs={}, response_metadata={}), AIMessage(content='The remains of Poseidonia are located in the modern frazione of Paestum, which is part of the comune of Capaccio Paestum in the Campania region of Italy. While the Greek settlers originally founded it, the Lucanians renamed the city to Paistos. The Romans later took over in 273 BCE, giving the city its current name, Paestum.', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-06-01T23:17:30.226607Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10972387292, 'load_duration': 104391667, 'prompt_eval_count': 1006, 'prompt_eval_duration': 1271783625, 'eval_count': 632, 'eval_duration': 9442151157, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019e857a-3514-7250-9835-f0909542250d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'inpu